In [1]:
!nvidia-smi

Sat Nov 21 20:40:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SAVE_DIR = "/content/drive/MyDrive/abliteration"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_ID = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
model.eval()

def build_prompt(user_msg):
    messages = [{"role": "user", "content": user_msg}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

Mounted at /content/drive


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (2 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
import random
import pandas as pd
from datasets import load_dataset

# Harmful: AdvBench (standard benchmark from the llm-attacks paper)
harmful_url = "https://raw.githubusercontent.com/llm-attacks/llm-attacks/main/data/advbench/harmful_behaviors.csv"
harmful_all = pd.read_csv(harmful_url)["goal"].tolist()

# Harmless: Alpaca instructions (only ones with no extra input)
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
harmless_all = [ex["instruction"] for ex in alpaca if ex["input"].strip() == ""]

random.seed(42)
random.shuffle(harmful_all)
random.shuffle(harmless_all)

N_TRAIN, N_TEST = 128, 32
harmful_train  = harmful_all[:N_TRAIN]
harmful_test   = harmful_all[N_TRAIN:N_TRAIN + N_TEST]
harmless_train = harmless_all[:N_TRAIN]
harmless_test  = harmless_all[N_TRAIN:N_TRAIN + N_TEST]

print(len(harmful_all), len(harmless_all))
print(harmless_train[:3])

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

520 31323
['Comment on the economic impact the pandemic has had on small businesses.', 'Change the idiom, “a piece of cake” to a more appropriate metaphor.', 'Tell me a story about a person working to create a sustainable future.']


In [ ]:
captured = {}

def make_hook(layer_idx):
    def hook(module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        captured[layer_idx] = h.detach()
    return hook

handle = model.model.layers[10].register_forward_hook(make_hook(10))

inputs = tokenizer(build_prompt("Who are you?"), return_tensors="pt").to(model.device)
with torch.no_grad():
    model(**inputs)

handle.remove()  # always remove hooks when done!

print("input_ids shape:", inputs.input_ids.shape)
print("layer 10 output shape:", captured[10].shape)

input_ids shape: torch.Size([1, 16])
layer 10 output shape: torch.Size([1, 16, 2048])


In [ ]:
last = captured[10][0, -1, :]   # 1st prompt, last token, all 2048 numbers
print(last.shape)               # torch.Size([2048])

torch.Size([2048])


In [3]:
tokenizer.padding_side = "left"

def collect_last_token_acts(prompts, batch_size=16):
    """Returns a tensor of shape [n_prompts, n_layers, d_model]:
    the residual stream at the last prompt token, after every decoder layer."""
    n_layers = len(model.model.layers)
    batch_acts = {}

    def make_hook(i):
        def hook(module, inputs, output):
            h = output[0] if isinstance(output, tuple) else output
            batch_acts[i] = h[:, -1, :].detach().float().cpu()
        return hook

    handles = [layer.register_forward_hook(make_hook(i))
               for i, layer in enumerate(model.model.layers)]
    all_acts = []
    try:
        for start in range(0, len(prompts), batch_size):
            texts = [build_prompt(p) for p in prompts[start:start + batch_size]]
            inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
            # positions count only real tokens, so left padding doesn't shift them
            position_ids = (inputs.attention_mask.cumsum(-1) - 1).clamp(min=0)
            with torch.no_grad():
                model(**inputs, position_ids=position_ids)
            all_acts.append(torch.stack([batch_acts[i] for i in range(n_layers)], dim=1))
    finally:
        for h in handles:
            h.remove()
    return torch.cat(all_acts, dim=0)

In [ ]:
harmful_acts  = collect_last_token_acts(harmful_train)
harmless_acts = collect_last_token_acts(harmless_train)
print("harmful:", harmful_acts.shape)
print("harmless:", harmless_acts.shape)

torch.save({
    "harmful_acts": harmful_acts,
    "harmless_acts": harmless_acts,
    "harmful_train": harmful_train, "harmful_test": harmful_test,
    "harmless_train": harmless_train, "harmless_test": harmless_test,
    "model_id": MODEL_ID,
}, f"{SAVE_DIR}/activations_train.pt")
print("saved to", SAVE_DIR)

harmful: torch.Size([128, 28, 2048])
harmless: torch.Size([128, 28, 2048])
saved to /content/drive/MyDrive/abliteration


In [4]:
saved = torch.load(f"{SAVE_DIR}/activations_train.pt")

harmful_test_acts  = collect_last_token_acts(saved["harmful_test"])
harmless_test_acts = collect_last_token_acts(saved["harmless_test"])
print(harmful_test_acts.shape, harmless_test_acts.shape)

torch.save({
    "harmful_test_acts": harmful_test_acts,
    "harmless_test_acts": harmless_test_acts,
}, f"{SAVE_DIR}/activations_test.pt")
print("saved")

torch.Size([32, 28, 2048]) torch.Size([32, 28, 2048])
saved
